# S2G Interactive Notebook
This notebook allows you to play around with the S2G model inputs, test out different schemas, and see the extracted knowledge graph in real time.

In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from s2g.linearisation import S2GTokens, add_special_tokens_to_tokenizer
from s2g.scripts.inference import extract_re, extract_boundary_joint

## Initialization
Load the model, tokenizer, and define schemas. Change the `checkpoint` variable to point to your trained model path or use `google/flan-t5-small` to test the untuned model.

In [ ]:
checkpoint = "google/flan-t5-base" # Replace with your trained checkpoint path
model_variant = "joint" # Change to "re", "boundary_re", "joint", or "boundary_joint"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

# Add S2G special tokens
tokens = S2GTokens(model_variant, use_rejection=False)
add_special_tokens_to_tokenizer(tokenizer, tokens, model)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6090.69it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- Corrected Model Output ---
['elder fühlt fühlt fühlt fühlt fühlt weil weil weil weil weildeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalbdeshalb']


## Define Schemas
You can either load schemas from a file, or define them manually as lists of strings.

In [ ]:
# Optionally load from file: 
# entity_schema = load_entity_schema("configs/schema/conll04_entities.json")
# rel_schema = load_schema("configs/schema/conll04_relations.json")

entity_schema = ["Loc", "Org", "Peop", "Other"]
rel_schema = ["Work_For", "Kill", "Organization_Based_In", "Live_In", "Located_In"]

## Inference
Run inference on a piece of text. We wrap the inference function based on the model variant.

In [ ]:
text = "John Wilkes Booth, who assassinated President Abraham Lincoln, was an actor from Maryland."

extract_fn = extract_re if model_variant in {"re", "boundary_re"} else extract_boundary_joint
tasks = [model_variant] # e.g. ["joint"]

result = extract_fn(
    text=text,
    model=model,
    tokenizer=tokenizer,
    entity_schema=entity_schema,
    rel_schema=rel_schema,
    tokens=tokens,
    num_beams=4,
    max_source_length=300,
    max_target_length=200,
    device=device,
    constraint_decoding=False,
    tasks=tasks,
    ssi_prompt="ssi" # Can be "ssi" or "natural"
)

print(json.dumps(result, indent=2))

## Raw Prompting
If you want to experiment with the model directly (providing the exact encoder string and viewing the raw string output without constraint decoding or parsing), use the cell below.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

# Optional: Force a completely clean cache folder if the download still fails
# os.environ["HF_HOME"] = "./hf_cache_fresh"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Force download to fix structural corruption, and set explicit bfloat16 precision
model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
).to(device)

Using device: cuda


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4054.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- Corrected Model Output ---
Palestinians killed in Israeli rebellion are featured in art exhibit


In [52]:
raw_input = (
    "Extract all entities of type [person, organization, location, other] and find relations of type "
    "[lives in, works for, is based in, killed, is located in] among the extracted entities in the given text."
    "Text: An art exhibit at the Hakawati Theatre in Arab east Jerusalem was a series of portraits of Palestinians killed in the rebellion ."
)

inputs = tokenizer(raw_input, return_tensors="pt").to(device)

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50  # Pure greedy search to verify model sanity
    )

# Use skip_special_tokens=True to remove <pad> and </s> tags
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("\n--- Corrected Model Output ---")
print(output_text)



--- Corrected Model Output ---
art exhibit , Palestinians , killed
